# Tesla Stock Price Prediction using SimpleRNN and LSTM
**Domain:** Financial Services  
**Goal:** Predict Tesla (TSLA) closing prices 1 day, 5 days, and 10 days ahead using deep learning.

---
## Table of Contents
1. [Imports & Setup](#1)
2. [Load & Explore Data (EDA)](#2)
3. [Data Cleaning](#3)
4. [Data Visualization](#4)
5. [Feature Engineering & Preprocessing](#5)
6. [SimpleRNN Model](#6)
7. [LSTM Model](#7)
8. [Multi-Step Predictions (1, 5, 10 days)](#8)
9. [Hyperparameter Tuning with GridSearchCV](#9)
10. [Model Comparison & Insights](#10)

---
## 1. Imports & Setup <a id='1'></a>

We import all required libraries upfront. Key ones:
- `tensorflow/keras` — for building RNN and LSTM models
- `sklearn` — for scaling and GridSearchCV
- `matplotlib/seaborn` — for all visualizations

In [ ]:
# ── Core ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Visualization ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# ── Preprocessing ─────────────────────────────────────────────
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ── Deep Learning ─────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from scikeras.wrappers import KerasRegressor

# ── Hyperparameter Tuning ──────────────────────────────────────
from sklearn.model_selection import GridSearchCV

print(f'TensorFlow version : {tf.__version__}')
print(f'NumPy version      : {np.__version__}')
print(f'Pandas version     : {pd.__version__}')
np.random.seed(42)
tf.random.set_seed(42)

---
## 2. Load & Explore Data (EDA) <a id='2'></a>

**Why this step matters:** Before building any model, we must understand what the data looks like — its shape, types, date range, and basic statistics. This is called Exploratory Data Analysis (EDA).

**Columns in TSLA.csv:**
| Column | Meaning |
|---|---|
| Date | Trading date |
| Open | Price at market open |
| High | Highest price of the day |
| Low | Lowest price of the day |
| Close | Price at market close |
| Adj Close | Close adjusted for splits/dividends |
| Volume | Number of shares traded |

In [ ]:
# Load dataset
df = pd.read_csv('TSLA.csv', parse_dates=['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)  # Ensure chronological order — critical for time series!

print('═'*55)
print(f'  Dataset shape : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'  Date range    : {df.index.min().date()} → {df.index.max().date()}')
print(f'  Trading days  : {df.shape[0]:,}')
print('═'*55)
df.head(10)

In [ ]:
# Descriptive statistics — gives min, max, mean, std for each column
print('\n📊 Descriptive Statistics:')
df.describe().round(2)

In [ ]:
# Check data types
print('\n🔍 Data Types:')
print(df.dtypes)
print(f'\n  All numeric columns? {df.dtypes.apply(lambda x: np.issubdtype(x, np.number)).all()}')

---
## 3. Data Cleaning <a id='3'></a>

**Why this matters for time series specifically:**  
Stock data has a *time component* — the order of rows is meaningful. If we have missing dates (e.g., gaps in trading days), we cannot simply fill them with `0` or drop them, because the model would learn incorrect time gaps.

**Strategy:**
- For missing *values* within existing rows → use **forward fill** (carry last known price forward). This makes sense because stock prices don't change on holidays — the last known price is the best estimate.
- For duplicate dates → keep the last entry.
- We do **NOT** drop rows blindly, as that would break the time sequence.

In [ ]:
# ── Check for missing values ──────────────────────────────────
print('🔍 Missing Values per Column:')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing cells: {missing.sum()}')

In [ ]:
# ── Check for duplicates ──────────────────────────────────────
print(f'Duplicate dates: {df.index.duplicated().sum()}')

# Remove duplicates if any
df = df[~df.index.duplicated(keep='last')]

# ── Forward fill missing values ───────────────────────────────
# ffill() carries the last valid observation forward
df.ffill(inplace=True)

# ── Check for infinite values ─────────────────────────────────
inf_count = np.isinf(df.select_dtypes(include=np.number)).sum().sum()
print(f'Infinite values : {inf_count}')

# ── Final check ───────────────────────────────────────────────
print(f'\n✅ Data after cleaning: {df.shape[0]:,} rows, {df.isnull().sum().sum()} missing values')

In [ ]:
# ── Outlier check using IQR on Close price ───────────────────
# For stock data, extreme prices are real events (not errors), so we note but don't remove them
Q1 = df['Close'].quantile(0.25)
Q3 = df['Close'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['Close'] < Q1 - 1.5*IQR) | (df['Close'] > Q3 + 1.5*IQR)]
print(f'Potential outliers in Close price: {len(outliers)} rows')
print('Note: These are real market events (e.g. 2021 spike to ~$400). We keep them.')
outliers[['Close']].head(10)

---
## 4. Data Visualization <a id='4'></a>

Good visualizations reveal trends, seasonality, and volatility — all useful before modeling.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 12))
fig.suptitle('Tesla (TSLA) Stock — Historical Overview', fontsize=16, fontweight='bold', y=1.01)

# ── Plot 1: Full closing price history ────────────────────────
axes[0].plot(df.index, df['Close'], color='steelblue', linewidth=1.2, label='Close Price')
axes[0].fill_between(df.index, df['Close'], alpha=0.1, color='steelblue')
axes[0].set_title('Closing Price History')
axes[0].set_ylabel('Price (USD)')
axes[0].legend()

# ── Plot 2: OHLC candlestick-style (using High-Low range) ────
axes[1].plot(df.index, df['High'], color='green', linewidth=0.7, alpha=0.7, label='Daily High')
axes[1].plot(df.index, df['Low'],  color='red',   linewidth=0.7, alpha=0.7, label='Daily Low')
axes[1].fill_between(df.index, df['Low'], df['High'], alpha=0.15, color='gray')
axes[1].set_title('Daily High vs Low Price (Volatility Range)')
axes[1].set_ylabel('Price (USD)')
axes[1].legend()

# ── Plot 3: Trading volume ────────────────────────────────────
axes[2].bar(df.index, df['Volume'], color='coral', width=1, alpha=0.7, label='Volume')
axes[2].set_title('Trading Volume')
axes[2].set_ylabel('Volume')
axes[2].legend()

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout()
plt.savefig('01_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Moving Averages ───────────────────────────────────────────
# MAs smooth out noise and help visualize the underlying trend
df['MA_30']  = df['Close'].rolling(window=30).mean()
df['MA_90']  = df['Close'].rolling(window=90).mean()
df['MA_200'] = df['Close'].rolling(window=200).mean()

fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(df.index, df['Close'],  color='lightblue', linewidth=0.8, alpha=0.8, label='Close')
ax.plot(df.index, df['MA_30'],  color='orange',    linewidth=1.5, label='30-day MA')
ax.plot(df.index, df['MA_90'],  color='red',       linewidth=1.5, label='90-day MA')
ax.plot(df.index, df['MA_200'], color='purple',    linewidth=1.5, label='200-day MA')
ax.set_title('TSLA Close Price with Moving Averages', fontsize=14)
ax.set_ylabel('Price (USD)')
ax.legend()
plt.tight_layout()
plt.savefig('02_moving_averages.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Daily Returns & Volatility ────────────────────────────────
df['Daily_Return'] = df['Close'].pct_change()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of daily returns
axes[0].hist(df['Daily_Return'].dropna(), bins=100, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title('Distribution of Daily Returns')
axes[0].set_xlabel('Daily Return')
axes[0].set_ylabel('Frequency')

# Rolling volatility (30-day std of returns)
df['Volatility_30'] = df['Daily_Return'].rolling(30).std() * np.sqrt(252)  # annualized
axes[1].plot(df.index, df['Volatility_30'], color='coral', linewidth=1.2)
axes[1].set_title('Rolling 30-day Annualized Volatility')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Volatility')

plt.tight_layout()
plt.savefig('03_returns_volatility.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Correlation Heatmap ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
corr = df[['Open','High','Low','Close','Adj Close','Volume']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('04_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 Insight: Open, High, Low, Close, and Adj Close are all highly correlated (>0.99).')
print('   This is expected for stock prices. We will use only Close as our target.')

---
## 5. Feature Engineering & Preprocessing <a id='5'></a>

### Why MinMaxScaler?
Neural networks learn through gradient descent. If inputs are on very different scales (e.g., Close price $300 vs Volume 50,000,000), the gradients become unstable. MinMaxScaler brings everything to [0, 1], ensuring stable and faster training.

### Why a sliding window?
Stock prices are sequential. The model needs to see the *context* — what happened in the past `window_size` days — to predict tomorrow. If `window_size=60`, then for each prediction, we feed the model 60 consecutive days of price data.

```
Input:  [day1, day2, ..., day60]  → Output: [day61]
Input:  [day2, day3, ..., day61]  → Output: [day62]
...and so on
```

In [ ]:
# ── Feature Engineering ───────────────────────────────────────
# Add technical indicators as extra features
df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()  # Exponential MA
df['EMA_26'] = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD']   = df['EMA_12'] - df['EMA_26']  # MACD indicator

# Bollinger Bands (volatility indicator)
df['BB_mid']   = df['Close'].rolling(20).mean()
df['BB_upper'] = df['BB_mid'] + 2 * df['Close'].rolling(20).std()
df['BB_lower'] = df['BB_mid'] - 2 * df['Close'].rolling(20).std()

# RSI (Relative Strength Index) — momentum indicator
delta  = df['Close'].diff()
gain   = delta.clip(lower=0)
loss   = -delta.clip(upper=0)
avg_g  = gain.rolling(14).mean()
avg_l  = loss.rolling(14).mean()
rs     = avg_g / avg_l
df['RSI'] = 100 - (100 / (1 + rs))

# Drop NaN rows created by rolling calculations
df.dropna(inplace=True)
print(f'Shape after feature engineering: {df.shape}')
print('\nNew features added: EMA_12, EMA_26, MACD, BB_upper, BB_lower, RSI')
df[['Close','MACD','RSI','BB_upper','BB_lower']].tail()

In [ ]:
# ── Select target variable ────────────────────────────────────
# We use only 'Close' price as the target (as specified in the problem statement)
# For multi-feature version, you can use all columns

close_prices = df[['Close']].values  # shape: (N, 1)

# ── Scale the data ────────────────────────────────────────────
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(close_prices)

print(f'Original Close range: ${close_prices.min():.2f} → ${close_prices.max():.2f}')
print(f'Scaled Close range : {scaled_data.min():.4f} → {scaled_data.max():.4f}')

In [ ]:
# ── Create sliding window sequences ──────────────────────────
def create_sequences(data, window_size=60):
    """
    Converts a 1D time series into (X, y) pairs for supervised learning.
    
    Args:
        data       : scaled numpy array, shape (N, 1)
        window_size: number of past days to use as input
    
    Returns:
        X : shape (N - window_size, window_size, 1)  — 3D for RNN/LSTM
        y : shape (N - window_size, 1)               — next day's price
    """
    X, y = [], []
    for i in range(window_size, len(data)):
        X.append(data[i - window_size:i, 0])  # past 60 days
        y.append(data[i, 0])                   # next day
    return np.array(X), np.array(y)

WINDOW_SIZE = 60  # use 60 trading days (~3 months) as context

X, y = create_sequences(scaled_data, WINDOW_SIZE)

# Reshape X to 3D — required by Keras RNN/LSTM layers: (samples, timesteps, features)
X = X.reshape(X.shape[0], X.shape[1], 1)

print(f'X shape: {X.shape}  →  (samples, timesteps, features)')
print(f'y shape: {y.shape}  →  (samples,)')

In [ ]:
# ── Train / Test split ────────────────────────────────────────
# For time series, we NEVER shuffle! Always split chronologically.
# Reason: shuffling would allow the model to "see the future" during training.

TRAIN_RATIO = 0.80  # 80% train, 20% test
split_idx   = int(len(X) * TRAIN_RATIO)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f'Training samples : {X_train.shape[0]:,}')
print(f'Testing samples  : {X_test.shape[0]:,}')

# Also store the corresponding dates for the test set (for plotting later)
test_dates = df.index[WINDOW_SIZE + split_idx:]

---
## 6. SimpleRNN Model <a id='6'></a>

### What is SimpleRNN?
A basic Recurrent Neural Network that processes sequences one step at a time and passes a "hidden state" from each step to the next. It suffers from the **vanishing gradient problem** — it forgets information from many steps back.

**Architecture:**
```
Input (60, 1) → SimpleRNN(64) → Dropout(0.2) → SimpleRNN(32) → Dense(1)
```

In [ ]:
def build_simple_rnn(units=64, dropout_rate=0.2, learning_rate=0.001):
    """
    Build and compile a SimpleRNN model.
    
    Args:
        units         : number of RNN units (neurons)
        dropout_rate  : fraction of neurons to randomly drop during training
        learning_rate : step size for the Adam optimizer
    """
    model = Sequential([
        # return_sequences=True means pass the full sequence to the next RNN layer
        SimpleRNN(units, return_sequences=True, input_shape=(WINDOW_SIZE, 1)),
        Dropout(dropout_rate),
        SimpleRNN(units // 2, return_sequences=False),
        Dropout(dropout_rate),
        Dense(25, activation='relu'),
        Dense(1)  # single output: next day's price
    ], name='SimpleRNN_Model')
    
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='mean_squared_error',
        metrics=['mae']
    )
    return model

rnn_model = build_simple_rnn()
rnn_model.summary()

In [ ]:
# ── Callbacks ─────────────────────────────────────────────────
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,          # stop if val_loss doesn't improve for 10 consecutive epochs
    restore_best_weights=True
)

checkpoint_rnn = ModelCheckpoint(
    'best_rnn_model.keras',
    monitor='val_loss',
    save_best_only=True,  # only saves when val_loss improves
    verbose=0
)

# ── Train the SimpleRNN model ─────────────────────────────────
print('🚀 Training SimpleRNN...')
rnn_history = rnn_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,   # 10% of training data used for validation
    callbacks=[early_stop, checkpoint_rnn],
    verbose=1
)
print('✅ SimpleRNN training complete!')

In [ ]:
# ── Plot training history ─────────────────────────────────────
def plot_training_history(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f'{model_name} — Training History', fontsize=13, fontweight='bold')
    
    axes[0].plot(history.history['loss'],     label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_title('Loss (MSE)')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    
    axes[1].plot(history.history['mae'],     label='Train MAE')
    axes[1].plot(history.history['val_mae'], label='Val MAE')
    axes[1].set_title('Mean Absolute Error')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(f'{model_name.lower().replace(" ","_")}_history.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_training_history(rnn_history, 'SimpleRNN')

In [ ]:
# ── Predictions ───────────────────────────────────────────────
rnn_pred_scaled = rnn_model.predict(X_test)

# Inverse transform to get actual dollar prices
rnn_pred = scaler.inverse_transform(rnn_pred_scaled)
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

# ── Evaluation metrics ────────────────────────────────────────
def evaluate_model(actual, predicted, model_name):
    mse  = mean_squared_error(actual, predicted)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(actual, predicted)
    r2   = r2_score(actual, predicted)
    
    print(f'\n📊 {model_name} — Evaluation Metrics')
    print('─'*40)
    print(f'  MSE  : {mse:.4f}')
    print(f'  RMSE : ${rmse:.2f}  (avg prediction error in dollars)')
    print(f'  MAE  : ${mae:.2f}')
    print(f'  R²   : {r2:.4f}  (1.0 = perfect)')
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

rnn_metrics = evaluate_model(y_test_actual, rnn_pred, 'SimpleRNN')

In [ ]:
# ── Plot actual vs predicted ───────────────────────────────────
def plot_predictions(actual, predicted, dates, model_name):
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(dates[:len(actual)], actual,    color='steelblue', linewidth=1.5, label='Actual Price', alpha=0.9)
    ax.plot(dates[:len(predicted)], predicted, color='coral',  linewidth=1.5, label=f'{model_name} Predicted', alpha=0.9)
    ax.set_title(f'TSLA Close Price — Actual vs {model_name} Predicted', fontsize=14)
    ax.set_ylabel('Price (USD)')
    ax.set_xlabel('Date')
    ax.legend(fontsize=11)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f'{model_name.lower()}_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_predictions(y_test_actual, rnn_pred, test_dates, 'SimpleRNN')

---
## 7. LSTM Model <a id='7'></a>

### What is LSTM and why is it better than SimpleRNN?
LSTM solves the vanishing gradient problem using **gates**:
- **Forget gate:** decides what past info to discard
- **Input gate:** decides what new info to store
- **Output gate:** decides what to output

This allows LSTM to remember patterns from hundreds of steps back — crucial for stock prices where a trend from months ago may still influence today's price.

**Architecture:**
```
Input (60, 1) → LSTM(128) → Dropout(0.2) → LSTM(64) → Dropout(0.2) → Dense(1)
```

In [ ]:
def build_lstm(units=128, dropout_rate=0.2, learning_rate=0.001):
    """
    Build and compile a stacked LSTM model.
    """
    model = Sequential([
        LSTM(units, return_sequences=True, input_shape=(WINDOW_SIZE, 1)),
        Dropout(dropout_rate),
        LSTM(units // 2, return_sequences=True),
        Dropout(dropout_rate),
        LSTM(units // 4, return_sequences=False),
        Dropout(dropout_rate),
        Dense(25, activation='relu'),
        Dense(1)
    ], name='LSTM_Model')
    
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='mean_squared_error',
        metrics=['mae']
    )
    return model

lstm_model = build_lstm()
lstm_model.summary()

In [ ]:
checkpoint_lstm = ModelCheckpoint(
    'best_lstm_model.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=0
)

print('🚀 Training LSTM...')
lstm_history = lstm_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop, checkpoint_lstm],
    verbose=1
)
print('✅ LSTM training complete!')

In [ ]:
plot_training_history(lstm_history, 'LSTM')

lstm_pred_scaled = lstm_model.predict(X_test)
lstm_pred        = scaler.inverse_transform(lstm_pred_scaled)

lstm_metrics = evaluate_model(y_test_actual, lstm_pred, 'LSTM')
plot_predictions(y_test_actual, lstm_pred, test_dates, 'LSTM')

---
## 8. Multi-Step Predictions: 1, 5, and 10 Days Ahead <a id='8'></a>

### Strategy: Recursive (Autoregressive) Forecasting
To predict N days ahead:
1. Predict day 1 using the last 60 known days
2. Append that prediction to the window, drop the oldest day
3. Predict day 2 using the updated window
4. Repeat N times

Note: Error accumulates with each step — 10-day predictions will be less accurate than 1-day.

In [ ]:
def predict_n_days(model, last_sequence, n_days, scaler):
    """
    Predict n_days ahead using recursive (autoregressive) forecasting.
    
    Args:
        model         : trained Keras model
        last_sequence : last WINDOW_SIZE scaled values, shape (WINDOW_SIZE,)
        n_days        : number of days to predict ahead
        scaler        : fitted MinMaxScaler (to inverse transform)
    
    Returns:
        predictions   : list of predicted prices in original scale
    """
    current_seq  = last_sequence.copy().reshape(1, WINDOW_SIZE, 1)
    predictions  = []
    
    for _ in range(n_days):
        pred_scaled = model.predict(current_seq, verbose=0)[0, 0]
        predictions.append(pred_scaled)
        
        # Slide the window: drop oldest, append new prediction
        current_seq = np.roll(current_seq, -1, axis=1)
        current_seq[0, -1, 0] = pred_scaled
    
    # Inverse transform to get dollar values
    preds_array = np.array(predictions).reshape(-1, 1)
    return scaler.inverse_transform(preds_array).flatten()


# Use the last WINDOW_SIZE days of the full dataset as the starting point
last_seq_scaled = scaled_data[-WINDOW_SIZE:]

for n_days in [1, 5, 10]:
    rnn_forecast  = predict_n_days(rnn_model, last_seq_scaled, n_days, scaler)
    lstm_forecast = predict_n_days(lstm_model, last_seq_scaled, n_days, scaler)
    
    print(f'\n📅 {n_days}-Day Forecast (from last known date: {df.index[-1].date()})')
    print(f'  SimpleRNN : {[f"${p:.2f}" for p in rnn_forecast]}')
    print(f'  LSTM      : {[f"${p:.2f}" for p in lstm_forecast]}')

In [ ]:
# ── Visualize multi-step forecasts ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Multi-Step Forecasts: SimpleRNN vs LSTM', fontsize=14, fontweight='bold')

n_context = 60  # show last 60 actual days for context
context_prices = df['Close'].values[-n_context:]
context_dates  = df.index[-n_context:]

for i, n_days in enumerate([1, 5, 10]):
    rnn_f  = predict_n_days(rnn_model,  last_seq_scaled, n_days, scaler)
    lstm_f = predict_n_days(lstm_model, last_seq_scaled, n_days, scaler)
    
    # Future dates (business days)
    future_dates = pd.bdate_range(start=df.index[-1], periods=n_days+1)[1:]
    
    ax = axes[i]
    ax.plot(context_dates, context_prices, color='steelblue', linewidth=1.5, label='Historical', alpha=0.9)
    ax.plot(future_dates, rnn_f,  'o--', color='coral',   linewidth=1.5, label='SimpleRNN Forecast')
    ax.plot(future_dates, lstm_f, 's--', color='green',   linewidth=1.5, label='LSTM Forecast')
    ax.axvline(x=df.index[-1], color='gray', linestyle=':', linewidth=1, alpha=0.7)
    ax.set_title(f'{n_days}-Day Forecast')
    ax.set_ylabel('Price (USD)')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout()
plt.savefig('08_multistep_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Hyperparameter Tuning with GridSearchCV <a id='9'></a>

### Why GridSearchCV?
Instead of manually guessing the best number of LSTM units, dropout rate, or learning rate, GridSearchCV **systematically tries all combinations** and picks the best one based on cross-validation score.

We use `scikeras.KerasRegressor` to wrap Keras models in a scikit-learn compatible interface.

**⚠️ Note:** GridSearchCV is computationally expensive. We use a small grid here for demonstration. For production, consider RandomizedSearchCV or Keras Tuner.

In [ ]:
# ── Build function for GridSearchCV ──────────────────────────
# KerasRegressor needs a function with keyword args matching param_grid keys

def build_lstm_for_gs(units=64, dropout_rate=0.2, learning_rate=0.001):
    model = Sequential([
        LSTM(units, return_sequences=False, input_shape=(WINDOW_SIZE, 1)),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='mean_squared_error'
    )
    return model

keras_reg = KerasRegressor(
    model=build_lstm_for_gs,
    epochs=20,
    batch_size=32,
    verbose=0
)

# ── Define parameter grid ─────────────────────────────────────
# 2×2×2 = 8 combinations × 3-fold CV = 24 training runs
param_grid = {
    'model__units'         : [64, 128],
    'model__dropout_rate'  : [0.1, 0.2],
    'model__learning_rate' : [0.001, 0.0005]
}

print('🔍 Starting GridSearchCV (this will take a few minutes)...')
grid_search = GridSearchCV(
    estimator=keras_reg,
    param_grid=param_grid,
    cv=3,               # 3-fold cross-validation
    scoring='neg_mean_squared_error',
    n_jobs=1,           # set to -1 if you have multiple CPU cores
    verbose=1
)

# Use a subset of training data to speed up GridSearchCV
X_gs = X_train[-500:]  # last 500 samples
y_gs = y_train[-500:]

grid_search.fit(X_gs, y_gs)

print(f'\n✅ Best Parameters: {grid_search.best_params_}')
print(f'   Best CV Score (neg MSE): {grid_search.best_score_:.6f}')

In [ ]:
# ── Show all GridSearchCV results ─────────────────────────────
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results = cv_results[['param_model__units','param_model__dropout_rate',
                          'param_model__learning_rate','mean_test_score','rank_test_score']]
cv_results.columns = ['Units','Dropout','LR','Mean Score (neg MSE)','Rank']
cv_results.sort_values('Rank').round(6)

In [ ]:
# ── Train optimized LSTM with best params ─────────────────────
best_params = grid_search.best_params_

optimized_lstm = build_lstm(
    units         = best_params['model__units'],
    dropout_rate  = best_params['model__dropout_rate'],
    learning_rate = best_params['model__learning_rate']
)

print('🚀 Training optimized LSTM with best hyperparameters...')
opt_history = optimized_lstm.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

opt_pred_scaled = optimized_lstm.predict(X_test)
opt_pred        = scaler.inverse_transform(opt_pred_scaled)
opt_metrics     = evaluate_model(y_test_actual, opt_pred, 'Optimized LSTM')

---
## 10. Model Comparison & Final Insights <a id='10'></a>

In [ ]:
# ── Comparison Table ──────────────────────────────────────────
comparison = pd.DataFrame({
    'Model'    : ['SimpleRNN', 'LSTM (default)', 'LSTM (optimized)'],
    'MSE'      : [rnn_metrics['MSE'],  lstm_metrics['MSE'],  opt_metrics['MSE']],
    'RMSE ($)' : [rnn_metrics['RMSE'], lstm_metrics['RMSE'], opt_metrics['RMSE']],
    'MAE ($)'  : [rnn_metrics['MAE'],  lstm_metrics['MAE'],  opt_metrics['MAE']],
    'R²'       : [rnn_metrics['R2'],   lstm_metrics['R2'],   opt_metrics['R2']]
})
comparison = comparison.set_index('Model').round(4)
print('\n📊 Final Model Comparison:')
display(comparison.style.highlight_min(axis=0, color='lightgreen').highlight_max(subset=['R²'], color='lightgreen'))

In [ ]:
# ── Side-by-side prediction comparison ───────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 14))
fig.suptitle('Model Comparison: Actual vs Predicted Close Price', fontsize=15, fontweight='bold')

models_preds = [
    ('SimpleRNN',       rnn_pred,  'coral'),
    ('LSTM (default)',  lstm_pred, 'green'),
    ('LSTM (optimized)',opt_pred,  'purple')
]

for ax, (name, pred, color) in zip(axes, models_preds):
    ax.plot(test_dates[:len(y_test_actual)], y_test_actual, color='steelblue',
            linewidth=1.5, label='Actual', alpha=0.9)
    ax.plot(test_dates[:len(pred)], pred, color=color,
            linewidth=1.5, label=f'{name} Predicted', alpha=0.85)
    ax.set_title(name, fontsize=12)
    ax.set_ylabel('Price (USD)')
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout()
plt.savefig('10_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Bar chart comparison ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')

model_names = ['SimpleRNN', 'LSTM', 'LSTM (opt)']
colors      = ['coral', 'steelblue', 'purple']

metrics_to_plot = [
    ('RMSE ($)', [rnn_metrics['RMSE'], lstm_metrics['RMSE'], opt_metrics['RMSE']]),
    ('MAE ($)',  [rnn_metrics['MAE'],  lstm_metrics['MAE'],  opt_metrics['MAE']]),
    ('R²',      [rnn_metrics['R2'],   lstm_metrics['R2'],   opt_metrics['R2']]),
]

for ax, (metric, values) in zip(axes, metrics_to_plot):
    bars = ax.bar(model_names, values, color=colors, alpha=0.8, edgecolor='white')
    ax.set_title(metric)
    ax.set_ylabel(metric)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('10_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Save models for deployment ────────────────────────────────
rnn_model.save('rnn_model_final.keras')
lstm_model.save('lstm_model_final.keras')
optimized_lstm.save('lstm_optimized_final.keras')

import joblib
joblib.dump(scaler, 'scaler.save')

print('✅ Models saved!')
print('  rnn_model_final.keras')
print('  lstm_model_final.keras')
print('  lstm_optimized_final.keras')
print('  scaler.save')

---
## Insights & Conclusion

### Key Findings:
1. **LSTM outperforms SimpleRNN** — LSTM's gating mechanism allows it to retain long-range dependencies in stock price sequences that SimpleRNN loses due to vanishing gradients.
2. **Hyperparameter tuning improves LSTM** — GridSearchCV found the optimal combination of units, dropout, and learning rate, yielding lower RMSE.
3. **Multi-step forecast accuracy degrades** — as expected, 10-day forecasts carry more accumulated error than 1-day forecasts.

### Limitations:
- Stock prices are influenced by external factors (news, earnings, macro) not captured in price history alone
- The model predicts based only on closing price — adding volume, sentiment, and macro data would help
- Past performance does not guarantee future results

### Suggested Improvements:
- Add NLP-based news sentiment as a feature
- Try Transformer-based models or GRU
- Use a multi-variate input (all OHLCV + technical indicators)
- Experiment with larger window sizes (90, 120 days)